# 电商用户行为分析

> 作者：___ ｜ 日期：2026-11-__ ｜ 一句话结论：___

**数据说明**：5000 名用户、400 个商品、约 4.6 万条订单（2024-01 ~ 2025-12），含少量缺失/重复/退货记录。


## 0. 环境与读数据


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['font.sans-serif'] = ['Microsoft YaHei']  # 解决中文显示
plt.rcParams['axes.unicode_minus'] = False

users = pd.read_csv('../data/users.csv')
products = pd.read_csv('../data/products.csv')
orders = pd.read_csv('../data/orders.csv', parse_dates=['order_date'])
orders.shape, users.shape, products.shape

((46348, 8), (5000, 6), (400, 3))

## 1. 数据体检（TODO）

要看四件事：行数、字段类型、缺失、重复。


In [3]:
# TODO: info / describe / isnull / duplicated
orders.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46348 entries, 0 to 46347
Data columns (total 8 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    46348 non-null  object        
 1   user_id     46348 non-null  object        
 2   product_id  46348 non-null  object        
 3   order_date  46348 non-null  datetime64[ns]
 4   quantity    46348 non-null  int64         
 5   amount      45885 non-null  float64       
 6   channel     46348 non-null  object        
 7   status      46348 non-null  object        
dtypes: datetime64[ns](1), float64(1), int64(1), object(5)
memory usage: 2.8+ MB


## 2. 清洗（TODO）

- 去重（order_id 重复）
- 金额缺失：先看缺失占比，再决定删还是补
- quantity < 0 是退货，单独标记，GMV 要剔除
- city 缺失填 '未知'


In [5]:
df = orders.copy()
print('原始', len(df))

# ① 去重：同一 order_id 只留一条
df = df.drop_duplicates(subset=['order_id'])
print('去重后', len(df))

# ② 金额缺失：先看有多少、是什么状态
print('金额缺失', df['amount'].isna().sum())
print(df[df['amount'].isna()]['status'].value_counts())

# ③ 退货：quantity < 0
df['is_return'] = df['quantity'] < 0
print('退货单', df['is_return'].sum())

# ④ 城市缺失
users['city'] = users['city'].fillna('未知')

# ⑤ 定义“有效订单”：已支付 + 非退货 + 金额不为空
paid = df[(df['status'] == '已支付') & (~df['is_return']) & df['amount'].notna()]
print('有效订单', len(paid))

原始 46348
去重后 46210
金额缺失 462
已支付    429
已退款     22
已取消     11
Name: status, dtype: int64
退货单 231
有效订单 41881


## 3. 核心指标（TODO，至少 4 个）

| 指标 | 口径 |
|---|---|
| GMV | 已支付订单金额之和 |
| 客单价 AOV | GMV / 支付订单数 |
| 复购率 | 下单≥2次用户 / 下单用户 |
| 月度环比 | (本月-上月)/上月 |


In [ ]:
# TODO: 先把每月 GMV 算出来，再画折线图


## 4. 图表（TODO）

至少 3 张：月度 GMV 趋势、城市 GMV 对比、品类占比。


In [ ]:
# TODO: 出图并 plt.savefig('../images/xxx.png', dpi=150, bbox_inches='tight')


## 5. 进阶：RFM + K-means 用户分群（选做，加分项）

1. 以 2025-12-31 为基准日，算 R（最近一次购买距今天数）、F（购买次数）、M（消费金额）
2. **标准化**（StandardScaler）——不标准化，金额会主导距离
3. 肘部法 + 轮廓系数选 k
4. KMeans 聚类，输出每类的 R/F/M 均值、人数占比
5. 给每类起名字并给业务建议


In [ ]:
# TODO: from sklearn.preprocessing import StandardScaler; from sklearn.cluster import KMeans


## 6. 结论与局限（面试官最爱问）

- 三条业务结论：
- 数据局限：
- 如果再做一次，我会改进：
